In [1]:
import numpy as np
#import time
import config.config as config
from validation import Validation
from Environment import Env
from Agent_EKF import *
from pathlib import Path

import sys; sys.path.append('../analysis/')
from my_utils import reset_seeds, cartesian_prod

In [2]:
def training(datapath, seed_number, task='gain'):
    # get configures
    if task == 'gain':
        arg = config.ConfigGain(datapath)
    elif 'noise' in task:
        arg = config.ConfigNoise(datapath)
    else:
        raise ValueError('No such a task!')
        
    arg.SEED_NUMBER = seed_number
    arg.save()
    
    # reproducibility
    reset_seeds(arg.SEED_NUMBER)
    
    # initialize environment and agent
    env = Env(arg)
    agent = Agent(arg)
    validator = Validation(arg.task, agent_name='EKF')
    
    # define exploration noise
    init_expnoise_std = 0.8
    noise = ActionNoise(arg.ACTION_DIM, mean=0, std=init_expnoise_std)

    # Remove observation noise in the beginning to help learning in the early stage.
    agent.bstep.obs_noise_range = None
    
    # Loop now
    tot_t = 0
    episode = agent.initial_episode
    reward_log = []
    rewarded_trial_log = []
    step_log = []
    actor_loss_log = 0
    critic_loss_log = 0
    dist_log = []
    
    LOG_FREQ = 100
    VALIDATION_FREQ = 500
    decrease_lr = True
    REPLAY_PERIOD = 4
    PRE_LEARN_PERIOD = arg.EKF_BATCH_SIZE * 50

    enable_mirror_traj = True
    pre_phase=True
    
    TOTAL_EPISODE = 1e5
    
    
    # Start loop
    while episode < TOTAL_EPISODE:
        # initialize a trial
        cross_start_threshold = False
        done = torch.zeros((1, 1), device=arg.device)
        reward = torch.zeros((1, 1), device=arg.device)
        
        x = env.reset()
        b, state = agent.bstep.reset(env.pro_gains, env.pro_noise_std, env.target_position)
        state = state.to(arg.device)
        last_action_raw = torch.zeros(1, arg.ACTION_DIM)

        for t in range(arg.EPISODE_LEN):
            # 1. Check start threshold.
            if not cross_start_threshold and (last_action_raw.abs() > arg.TERMINAL_ACTION).any():
                cross_start_threshold = True

            # 2. Take an action based on current state .
            action, action_raw = agent.select_action(state, action_noise=noise)

            # 3. Track next x in the environment.
            next_x, reached_target, relative_dist = env(x, action, t)

            # 4. Next belief state given next x.
            next_ox = agent.bstep.observation(next_x)
            next_b = agent.bstep(b, next_ox, action, env.perturbation_vt, env.perturbation_wt)
            next_state = agent.bstep.b_reshape(next_b).to(arg.device)

            # 5. Check whether stop.
            time_end = (t == arg.EPISODE_LEN - 1)
            is_stop = env.is_stop(x, action)

            # 6. Give reward if stopped.
            if (is_stop and cross_start_threshold) or time_end:
                done = torch.ones((1, 1), device=arg.device)
                if is_stop and cross_start_threshold:
                    reward = env.return_reward(x, reward_mode='mixed').view(-1, 1).to(arg.device)

            # 7. Push data into buffer.
            agent.memory.push(state, action.to(arg.device), next_state, reward, done)
            if enable_mirror_traj and noise.std != init_expnoise_std:
                agent.memory.push(*agent.mirror_traj(state, action, next_state), reward, done)      

            # 8. Update timestep.
            last_action_raw = action_raw
            state = next_state
            x = next_x
            b = next_b 
            tot_t += 1

            # 9. Update model.
            if len(agent.memory.memory) > PRE_LEARN_PERIOD and tot_t % REPLAY_PERIOD == 0:
                #start_time = time.time()
                actor_loss, critic_loss = agent.learn()
                actor_loss_log += actor_loss
                critic_loss_log += critic_loss
                #time_all.append(time.time() - start_time)

            # 10. whether break.
            if is_stop and cross_start_threshold:
                break


        # End of one trial.
        # log stuff        
        reward_log.append(reward.item())
        rewarded_trial_log.append(int(reached_target & is_stop))
        step_log.append(t + 1)
        dist_log.append(relative_dist.item())

        if episode % LOG_FREQ == LOG_FREQ - 1:
            print(f"t: {tot_t}, Ep: {episode}, action std: {noise.std:0.2f}")
            print(f"mean steps: {np.mean(step_log):0.3f}, "
                  f"mean reward: {np.mean(reward_log):0.3f}, "
                  f"rewarded fraction: {np.mean(rewarded_trial_log):0.3f}, "
                  f"relative distance: {np.mean(dist_log) * arg.LINEAR_SCALE:0.3f}, "
                  f"obs noise: {agent.bstep.obs_noise_range}")

            if decrease_lr and (validator.data.reward_fraction > 0.95).any():
                noise.reset(mean=0, std=0.4)
                agent.actor_optim.param_groups[0]['lr'] = arg.decayed_lr
                agent.critic_optim.param_groups[0]['lr'] = arg.decayed_lr
                decrease_lr = False
                print('Noise and learning rate are changed.')

            if noise.std == init_expnoise_std and np.mean(rewarded_trial_log) > 0.2:
                noise.reset(mean=0, std=0.5)
                agent.bstep.obs_noise_range = arg.obs_noise_range
                agent.memory.reset()
                tot_t = 0
                episode = 0
                pre_phase = False

            reward_log = []
            rewarded_trial_log = []
            step_log = []
            actor_loss_log = 0
            critic_loss_log = 0
            dist_log = []

        # saving and validation
        if episode % VALIDATION_FREQ == VALIDATION_FREQ - 1:
            # save
            agent.save(save_memory=False, episode=episode, pre_phase=pre_phase)

            if noise.std < init_expnoise_std and decrease_lr:
            #if noise.std < init_expnoise_std:
                validator(agent, episode).to_csv(arg.data_path / f'{arg.filename}.csv', index=False)
                agent.bstep.obs_noise_range = arg.obs_noise_range

        episode += 1

In [3]:
task = 'noise'
seeds = [7]
for seed in seeds:
    datapath = Path.cwd().parents[1] / 'agents' / 'EKF' / task / f'seed{seed}'
    training(datapath, seed, task)

t: 664, Ep: 99, action std: 0.80
mean steps: 6.640, mean reward: 0.028, rewarded fraction: 0.000, relative distance: 268.726, obs noise: [0, 0]
t: 1283, Ep: 199, action std: 0.80
mean steps: 6.190, mean reward: 0.392, rewarded fraction: 0.030, relative distance: 255.007, obs noise: [0, 0]
t: 1807, Ep: 299, action std: 0.80
mean steps: 5.240, mean reward: 0.052, rewarded fraction: 0.000, relative distance: 270.413, obs noise: [0, 0]
t: 2406, Ep: 399, action std: 0.80
mean steps: 5.990, mean reward: 0.258, rewarded fraction: 0.020, relative distance: 261.097, obs noise: [0, 0]
t: 2921, Ep: 499, action std: 0.80
mean steps: 5.150, mean reward: 0.372, rewarded fraction: 0.030, relative distance: 254.804, obs noise: [0, 0]
t: 3492, Ep: 599, action std: 0.80
mean steps: 5.710, mean reward: 0.165, rewarded fraction: 0.010, relative distance: 272.816, obs noise: [0, 0]
t: 4150, Ep: 699, action std: 0.80
mean steps: 6.580, mean reward: 0.043, rewarded fraction: 0.000, relative distance: 270.205

t: 118133, Ep: 5699, action std: 0.80
mean steps: 33.800, mean reward: 0.026, rewarded fraction: 0.000, relative distance: 390.031, obs noise: [0, 0]
t: 121451, Ep: 5799, action std: 0.80
mean steps: 33.180, mean reward: 0.007, rewarded fraction: 0.000, relative distance: 381.969, obs noise: [0, 0]
t: 124782, Ep: 5899, action std: 0.80
mean steps: 33.310, mean reward: 0.011, rewarded fraction: 0.000, relative distance: 373.105, obs noise: [0, 0]
t: 128149, Ep: 5999, action std: 0.80
mean steps: 33.670, mean reward: 0.002, rewarded fraction: 0.000, relative distance: 403.479, obs noise: [0, 0]
t: 131519, Ep: 6099, action std: 0.80
mean steps: 33.700, mean reward: 0.107, rewarded fraction: 0.010, relative distance: 398.717, obs noise: [0, 0]
t: 134918, Ep: 6199, action std: 0.80
mean steps: 33.990, mean reward: 0.101, rewarded fraction: 0.010, relative distance: 392.126, obs noise: [0, 0]
t: 138307, Ep: 6299, action std: 0.80
mean steps: 33.890, mean reward: 0.001, rewarded fraction: 0.0

t: 303370, Ep: 11199, action std: 0.80
mean steps: 32.820, mean reward: 0.001, rewarded fraction: 0.000, relative distance: 321.999, obs noise: [0, 0]
t: 306773, Ep: 11299, action std: 0.80
mean steps: 34.030, mean reward: 0.014, rewarded fraction: 0.000, relative distance: 338.964, obs noise: [0, 0]
t: 310233, Ep: 11399, action std: 0.80
mean steps: 34.600, mean reward: 0.011, rewarded fraction: 0.000, relative distance: 345.609, obs noise: [0, 0]
t: 313616, Ep: 11499, action std: 0.80
mean steps: 33.830, mean reward: 0.008, rewarded fraction: 0.000, relative distance: 324.061, obs noise: [0, 0]
t: 316893, Ep: 11599, action std: 0.80
mean steps: 32.770, mean reward: 0.013, rewarded fraction: 0.000, relative distance: 312.383, obs noise: [0, 0]
t: 320214, Ep: 11699, action std: 0.80
mean steps: 33.210, mean reward: 0.012, rewarded fraction: 0.000, relative distance: 327.887, obs noise: [0, 0]
t: 323530, Ep: 11799, action std: 0.80
mean steps: 33.160, mean reward: 0.013, rewarded fracti

t: 487749, Ep: 16699, action std: 0.80
mean steps: 33.580, mean reward: 0.106, rewarded fraction: 0.010, relative distance: 299.585, obs noise: [0, 0]
t: 491057, Ep: 16799, action std: 0.80
mean steps: 33.080, mean reward: 0.169, rewarded fraction: 0.010, relative distance: 300.300, obs noise: [0, 0]
t: 494434, Ep: 16899, action std: 0.80
mean steps: 33.770, mean reward: 0.007, rewarded fraction: 0.000, relative distance: 308.941, obs noise: [0, 0]
t: 497889, Ep: 16999, action std: 0.80
mean steps: 34.550, mean reward: 0.000, rewarded fraction: 0.000, relative distance: 334.692, obs noise: [0, 0]
t: 501326, Ep: 17099, action std: 0.80
mean steps: 34.370, mean reward: 0.001, rewarded fraction: 0.000, relative distance: 346.936, obs noise: [0, 0]
t: 504650, Ep: 17199, action std: 0.80
mean steps: 33.240, mean reward: 0.026, rewarded fraction: 0.000, relative distance: 341.290, obs noise: [0, 0]
t: 507951, Ep: 17299, action std: 0.80
mean steps: 33.010, mean reward: 0.012, rewarded fracti

t: 57606, Ep: 1899, action std: 0.50
mean steps: 28.410, mean reward: 3.856, rewarded fraction: 0.360, relative distance: 58.482, obs noise: [0, 1]
t: 60298, Ep: 1999, action std: 0.50
mean steps: 26.920, mean reward: 3.854, rewarded fraction: 0.350, relative distance: 62.874, obs noise: [0, 1]
t: 62378, Ep: 2099, action std: 0.50
mean steps: 20.800, mean reward: 1.431, rewarded fraction: 0.100, relative distance: 129.910, obs noise: [0, 1]
t: 64473, Ep: 2199, action std: 0.50
mean steps: 20.950, mean reward: 1.892, rewarded fraction: 0.140, relative distance: 126.973, obs noise: [0, 1]
t: 66385, Ep: 2299, action std: 0.50
mean steps: 19.120, mean reward: 4.661, rewarded fraction: 0.410, relative distance: 79.430, obs noise: [0, 1]
t: 68196, Ep: 2399, action std: 0.50
mean steps: 18.110, mean reward: 7.160, rewarded fraction: 0.660, relative distance: 55.498, obs noise: [0, 1]
t: 70064, Ep: 2499, action std: 0.50
mean steps: 18.680, mean reward: 9.030, rewarded fraction: 0.890, relativ

...

t: 1308464, Ep: 73399, action std: 0.40
mean steps: 17.660, mean reward: 9.527, rewarded fraction: 0.940, relative distance: 35.276, obs noise: [0, 1]
t: 1310259, Ep: 73499, action std: 0.40
mean steps: 17.950, mean reward: 9.159, rewarded fraction: 0.900, relative distance: 38.377, obs noise: [0, 1]
t: 1312020, Ep: 73599, action std: 0.40
mean steps: 17.610, mean reward: 9.463, rewarded fraction: 0.930, relative distance: 34.963, obs noise: [0, 1]
t: 1313716, Ep: 73699, action std: 0.40
mean steps: 16.960, mean reward: 9.439, rewarded fraction: 0.930, relative distance: 36.161, obs noise: [0, 1]
t: 1315496, Ep: 73799, action std: 0.40
mean steps: 17.800, mean reward: 9.509, rewarded fraction: 0.940, relative distance: 32.469, obs noise: [0, 1]
t: 1317129, Ep: 73899, action std: 0.40
mean steps: 16.330, mean reward: 9.042, rewarded fraction: 0.890, relative distance: 42.162, obs noise: [0, 1]
t: 1318808, Ep: 73999, action std: 0.40
mean steps: 16.790, mean reward: 9.543, rewarded fract

t: 1404339, Ep: 78899, action std: 0.40
mean steps: 17.330, mean reward: 9.084, rewarded fraction: 0.890, relative distance: 38.847, obs noise: [0, 1]
t: 1406129, Ep: 78999, action std: 0.40
mean steps: 17.900, mean reward: 8.822, rewarded fraction: 0.860, relative distance: 41.546, obs noise: [0, 1]
t: 1407888, Ep: 79099, action std: 0.40
mean steps: 17.590, mean reward: 9.108, rewarded fraction: 0.890, relative distance: 37.870, obs noise: [0, 1]
t: 1409667, Ep: 79199, action std: 0.40
mean steps: 17.790, mean reward: 9.677, rewarded fraction: 0.960, relative distance: 32.004, obs noise: [0, 1]
t: 1411393, Ep: 79299, action std: 0.40
mean steps: 17.260, mean reward: 9.204, rewarded fraction: 0.900, relative distance: 35.615, obs noise: [0, 1]
t: 1413186, Ep: 79399, action std: 0.40
mean steps: 17.930, mean reward: 9.443, rewarded fraction: 0.930, relative distance: 39.041, obs noise: [0, 1]
t: 1414955, Ep: 79499, action std: 0.40
mean steps: 17.690, mean reward: 9.752, rewarded fract

t: 1501212, Ep: 84399, action std: 0.40
mean steps: 16.800, mean reward: 9.502, rewarded fraction: 0.940, relative distance: 37.997, obs noise: [0, 1]
t: 1502926, Ep: 84499, action std: 0.40
mean steps: 17.140, mean reward: 9.274, rewarded fraction: 0.910, relative distance: 38.683, obs noise: [0, 1]
t: 1504670, Ep: 84599, action std: 0.40
mean steps: 17.440, mean reward: 9.606, rewarded fraction: 0.950, relative distance: 36.830, obs noise: [0, 1]
t: 1506464, Ep: 84699, action std: 0.40
mean steps: 17.940, mean reward: 9.343, rewarded fraction: 0.920, relative distance: 37.918, obs noise: [0, 1]
t: 1508144, Ep: 84799, action std: 0.40
mean steps: 16.800, mean reward: 9.368, rewarded fraction: 0.930, relative distance: 43.665, obs noise: [0, 1]
t: 1509916, Ep: 84899, action std: 0.40
mean steps: 17.720, mean reward: 9.642, rewarded fraction: 0.960, relative distance: 35.764, obs noise: [0, 1]
t: 1511688, Ep: 84999, action std: 0.40
mean steps: 17.720, mean reward: 9.155, rewarded fract

t: 1598631, Ep: 89899, action std: 0.40
mean steps: 17.020, mean reward: 9.292, rewarded fraction: 0.910, relative distance: 37.959, obs noise: [0, 1]
t: 1600400, Ep: 89999, action std: 0.40
mean steps: 17.690, mean reward: 9.440, rewarded fraction: 0.930, relative distance: 38.331, obs noise: [0, 1]
t: 1602161, Ep: 90099, action std: 0.40
mean steps: 17.610, mean reward: 9.372, rewarded fraction: 0.920, relative distance: 37.330, obs noise: [0, 1]
t: 1603918, Ep: 90199, action std: 0.40
mean steps: 17.570, mean reward: 8.676, rewarded fraction: 0.840, relative distance: 41.105, obs noise: [0, 1]
t: 1605619, Ep: 90299, action std: 0.40
mean steps: 17.010, mean reward: 9.552, rewarded fraction: 0.940, relative distance: 34.866, obs noise: [0, 1]
t: 1607398, Ep: 90399, action std: 0.40
mean steps: 17.790, mean reward: 9.225, rewarded fraction: 0.900, relative distance: 36.016, obs noise: [0, 1]
t: 1609139, Ep: 90499, action std: 0.40
mean steps: 17.410, mean reward: 9.613, rewarded fract

t: 1695128, Ep: 95399, action std: 0.40
mean steps: 17.040, mean reward: 9.709, rewarded fraction: 0.960, relative distance: 33.909, obs noise: [0, 1]
t: 1696885, Ep: 95499, action std: 0.40
mean steps: 17.570, mean reward: 9.618, rewarded fraction: 0.950, relative distance: 34.538, obs noise: [0, 1]
t: 1698622, Ep: 95599, action std: 0.40
mean steps: 17.370, mean reward: 9.124, rewarded fraction: 0.890, relative distance: 40.459, obs noise: [0, 1]
t: 1700457, Ep: 95699, action std: 0.40
mean steps: 18.350, mean reward: 8.956, rewarded fraction: 0.870, relative distance: 37.477, obs noise: [0, 1]
t: 1702326, Ep: 95799, action std: 0.40
mean steps: 18.690, mean reward: 9.360, rewarded fraction: 0.920, relative distance: 34.729, obs noise: [0, 1]
t: 1704012, Ep: 95899, action std: 0.40
mean steps: 16.860, mean reward: 9.193, rewarded fraction: 0.900, relative distance: 40.563, obs noise: [0, 1]
t: 1705790, Ep: 95999, action std: 0.40
mean steps: 17.780, mean reward: 9.357, rewarded fract